# Лабораторна робота №2 — Частина 2

## Individual Household Electric Power Consumption Dataset

У цій частині роботи виконується очищення та аналіз датасету споживання електроенергії домогосподарством.


## Завдання 1

Завантажити та відкрити датасет **Individual Household Electric Power Consumption Dataset**.

Датасет потрібно завантажити вручну з UCI Machine Learning Repository або іншого офіційного джерела, розпакувати файл і покласти файл `household_power_consumption.txt` у директорію:

```text
lab_02/data/household_power_consumption.txt
```

Файл з даними не додається до GitHub, оскільки він великий.


In [ ]:
from pathlib import Path
import timeit

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

DATA_PATH = Path("data/household_power_consumption.txt")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Файл з датасетом не знайдено. "
        "Завантажте household_power_consumption.txt і покладіть його у lab_02/data/."
    )


## Завдання 2

Зчитати датасет у pandas DataFrame та виконати data cleaning.

У цьому датасеті пропущені значення позначені символом `?`, тому під час зчитування вони перетворюються на `NaN`.


In [ ]:
def load_power_consumption_dataset(path: Path) -> pd.DataFrame:
    """Завантажує та очищує датасет споживання електроенергії."""
    df = pd.read_csv(
        path,
        sep=";",
        na_values="?",
        low_memory=False,
    )
    
    df["datetime"] = pd.to_datetime(
        df["Date"] + " " + df["Time"],
        format="%d/%m/%Y %H:%M:%S",
        errors="coerce",
    )
    
    numeric_columns = [
        "Global_active_power",
        "Global_reactive_power",
        "Voltage",
        "Global_intensity",
        "Sub_metering_1",
        "Sub_metering_2",
        "Sub_metering_3",
    ]
    
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    
    df = df.dropna(subset=["datetime"])
    df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].median())
    
    df["date"] = df["datetime"].dt.date
    df["time"] = df["datetime"].dt.time
    df["hour"] = df["datetime"].dt.hour
    df["month"] = df["datetime"].dt.month
    df["day_of_week"] = df["datetime"].dt.day_name()
    
    return df

power_df = load_power_consumption_dataset(DATA_PATH)
power_df.head()


In [ ]:
power_df.info()


## Завдання 3

Окремими функціями сформувати вибірки згідно з умовою лабораторної роботи.


In [ ]:
def select_active_power_over_5kw(df: pd.DataFrame) -> pd.DataFrame:
    """Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт."""
    return df[df["Global_active_power"] > 5].copy()

def select_intensity_19_20_and_submetering_condition(df: pd.DataFrame) -> pd.DataFrame:
    """
    Обрати записи, у яких сила струму лежить в межах 19–20 А.
    Для них вибрати ті, де Sub_metering_2 і Sub_metering_3 разом більші за Sub_metering_1.
    
    Інтерпретація груп:
    - Sub_metering_1: кухня / бойлер та кондиціонер згідно з умовою;
    - Sub_metering_2 і Sub_metering_3: пральна машина, холодильник, освітлення тощо.
    """
    filtered = df[df["Global_intensity"].between(19, 20, inclusive="both")].copy()
    condition = (filtered["Sub_metering_2"] + filtered["Sub_metering_3"]) > filtered["Sub_metering_1"]
    return filtered[condition].copy()

def random_sample_500000_and_mean_submetering(df: pd.DataFrame, random_state: int = 42) -> pd.Series:
    """Обрати випадкові 500000 записів без повторів і обчислити середні для 3 груп споживання."""
    sample_size = min(500_000, len(df))
    sample = df.sample(n=sample_size, replace=False, random_state=random_state)
    return sample[["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]].mean()

def select_evening_high_consumption_complex(df: pd.DataFrame) -> pd.DataFrame:
    """
    Обрати записи після 18:00 з потужністю понад 6 кВт.
    Серед них залишити записи, де Sub_metering_2 є найбільшою групою.
    Потім обрати кожен третій результат із першої половини та кожен четвертий із другої половини.
    """
    filtered = df[(df["hour"] >= 18) & (df["Global_active_power"] > 6)].copy()
    filtered = filtered[
        (filtered["Sub_metering_2"] > filtered["Sub_metering_1"])
        & (filtered["Sub_metering_2"] > filtered["Sub_metering_3"])
    ].copy()
    
    midpoint = len(filtered) // 2
    first_half = filtered.iloc[:midpoint].iloc[::3]
    second_half = filtered.iloc[midpoint:].iloc[::4]
    
    return pd.concat([first_half, second_half]).reset_index(drop=True)

def normalize_and_standardize(df: pd.DataFrame, columns: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Повертає нормалізований і стандартизований DataFrame для вказаних колонок."""
    minmax_scaler = MinMaxScaler()
    standard_scaler = StandardScaler()
    
    normalized = pd.DataFrame(
        minmax_scaler.fit_transform(df[columns]),
        columns=[f"{col}_normalized" for col in columns],
        index=df.index,
    )
    
    standardized = pd.DataFrame(
        standard_scaler.fit_transform(df[columns]),
        columns=[f"{col}_standardized" for col in columns],
        index=df.index,
    )
    
    return normalized, standardized

def calculate_correlations(df: pd.DataFrame, column_1: str, column_2: str) -> pd.Series:
    """Підрахувати коефіцієнти Пірсона та Спірмена для двох числових атрибутів."""
    return pd.Series(
        {
            "pearson": df[column_1].corr(df[column_2], method="pearson"),
            "spearman": df[column_1].corr(df[column_2], method="spearman"),
        }
    )

def one_hot_encode_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Виконати One Hot Encoding категоріального атрибута."""
    return pd.get_dummies(df, columns=[column], prefix=column)


## Завдання 4

Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.


In [ ]:
result_1 = select_active_power_over_5kw(power_df)
result_1.head(), result_1.shape


## Завдання 5

Обрати всі записи, у яких сила струму лежить в межах 19–20 А, і відібрати ті, де одна група споживання переважає іншу згідно з умовою.


In [ ]:
result_2 = select_intensity_19_20_and_submetering_condition(power_df)
result_2.head(), result_2.shape


## Завдання 6

Обрати випадковим чином 500000 записів без повторів та обчислити середні величини усіх 3-х груп споживання електроенергії.


In [ ]:
result_3 = random_sample_500000_and_mean_submetering(power_df)
result_3


## Завдання 7

Обрати записи після 18:00 з потужністю понад 6 кВт, залишити записи з найбільшим значенням другої групи споживання, а потім вибрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.


In [ ]:
result_4 = select_evening_high_consumption_complex(power_df)
result_4.head(), result_4.shape


## Завдання 8

Пронормувати та стандартизувати вибраний датасет.


In [ ]:
numeric_columns_for_scaling = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]

normalized_df, standardized_df = normalize_and_standardize(power_df, numeric_columns_for_scaling)
normalized_df.head()


In [ ]:
standardized_df.head()


## Завдання 9

Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.


In [ ]:
calculate_correlations(power_df, "Global_active_power", "Global_intensity")


## Завдання 10

Провести One Hot Encoding категоріального атрибута.

Категоріальним атрибутом у цій роботі буде `day_of_week`, який ми отримали з дати.


In [ ]:
encoded_sample = one_hot_encode_column(power_df.head(20), "day_of_week")
encoded_sample.head()


## Завдання 11

Проаналізувати часові витрати на виконання процедур за допомогою модуля `timeit`.


In [ ]:
def profile_function(function_name: str, statement: str, globals_dict: dict, number: int = 3) -> float:
    """Повертає середній час виконання statement."""
    total_time = timeit.timeit(statement, globals=globals_dict, number=number)
    return total_time / number

profile_results = pd.DataFrame(
    [
        {
            "operation": "Global_active_power > 5",
            "avg_time_seconds": profile_function(
                "select_active_power_over_5kw",
                "select_active_power_over_5kw(power_df)",
                globals(),
            ),
        },
        {
            "operation": "Global_intensity 19-20 + condition",
            "avg_time_seconds": profile_function(
                "select_intensity_19_20_and_submetering_condition",
                "select_intensity_19_20_and_submetering_condition(power_df)",
                globals(),
            ),
        },
        {
            "operation": "Random sample 500000 + mean",
            "avg_time_seconds": profile_function(
                "random_sample_500000_and_mean_submetering",
                "random_sample_500000_and_mean_submetering(power_df)",
                globals(),
            ),
        },
        {
            "operation": "Evening high consumption complex selection",
            "avg_time_seconds": profile_function(
                "select_evening_high_consumption_complex",
                "select_evening_high_consumption_complex(power_df)",
                globals(),
            ),
        },
    ]
)

profile_results
